# IMDB Sentiment Prediction — Project Workflow

### Goal
Use the Simple RNN trained in `simplernn.ipynb` (saved as `simple_rnn_imdb.h5`) to predict whether a **new, user-written movie review** is **Positive** or **Negative**. No training happens in this notebook; it only loads the saved model and uses it for inference.

### Workflow

```
Review text -> clean + split into words -> word IDs (+3 offset) -> pad to 500 -> (1, 500) array
            -> load model -> model.predict -> score between 0 and 1 -> > 0.5 ? Positive : Negative
```

| Stage | What happens | Notebook steps |
|---|---|---|
| 1. Setup | Import NumPy, TensorFlow, the IMDB dataset helper, `sequence` and `load_model`. | Step 1 |
| 2. Get the word dictionary | `imdb.get_word_index()` gives word -> ID (the same dictionary used in training) and the reverse ID -> word. | Step 2 |
| 3. Load the trained model | `load_model('simple_rnn_imdb.h5')` restores the architecture and learned weights; `summary()` confirms it. | Step 3 |
| 4. (Optional) Inspect weights | `model.get_weights()` shows the trained values; only for exploration. | Step 4 |
| 5. Preprocess text | `decode_review` (IDs -> text) and `preprocess_text` (text -> lower-case words -> IDs + 3 -> pad to 500). | Step 5 |
| 6. Predict | `predict_sentiment` preprocesses the text, runs `model.predict` and applies the 0.5 threshold. | Step 6 |
| 7. Try an example | Run the function on a sample review and print the sentiment and score. | Step 7 |
| 8. Improve preprocessing | A regex version removes punctuation and HTML tags and maps unknown words to the correct ID `2`. | Step 8 |

### Why preprocessing must match training
The model learned from reviews encoded in a specific way, so a new review has to be encoded **exactly the same way**:
- the **same word dictionary** (`word_index`),
- the **same +3 offset** (IDs 0, 1, 2 are reserved for padding, start and unknown),
- the **same length of 500** (padding with `0`s at the front),
- the **same vocabulary limit** of 10,000 words (anything rarer becomes the unknown ID `2`).

If any of these differ, the IDs point to the wrong words/embedding rows and the predictions become unreliable.

### Input and output shapes
```
Text            "This movie was fantastic!"
Word IDs        [14, 20, 16, 4098]               (variable length)
After padding   (1, 500)                         1 review x 500 IDs
Model output    (1, 1)                           one probability
Final answer    Positive / Negative + score
```

### Reading the result
- Score close to **1** -> strongly positive; close to **0** -> strongly negative; near **0.5** -> uncertain.
- Example in this notebook: score about `0.81` -> **Positive**.

### How this connects to the rest of the project
`embedding.ipynb` (concepts) -> `simplernn.ipynb` (train and save the model) -> **`prediction.ipynb`** (this notebook: predict new reviews) -> `main.py` (Streamlit app that uses the same preprocessing and the same saved model).

### Requirements
`simple_rnn_imdb.h5` must be in the same folder as this notebook, and `tensorflow` (2.15 in `requirements.txt`) and `numpy` must be installed.

### Step 1 — Import libraries

**What this cell does:** imports the packages needed to load the trained model and prepare new text.
- `numpy`, `tensorflow` — numerical and deep learning libraries.
- `imdb` — we only need it to get the word-to-ID dictionary used during training.
- `sequence` — for padding the new review to the right length.
- `load_model` — loads a model that was previously saved to disk.

**Note:** the output you see is only a TensorFlow deprecation warning and is harmless.

In [10]:
# Step 1: Import Libraries and Load the Model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

### Step 2 — Load the IMDB word index

**What this cell does:**
1. `imdb.get_word_index()` returns the dictionary **word → integer ID** used when the model was trained.
2. `reverse_word_index` flips it to **integer ID → word**.

**Why it is needed:** the model only understands integers. To classify a new review we must convert its words using exactly the *same* dictionary the training data used, otherwise the IDs would not match what the model learned.

In [11]:

# Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [12]:
reverse_word_index

{34701: 'fawn',
 52006: 'tsukino',
 52007: 'nunnery',
 16816: 'sonja',
 63951: 'vani',
 1408: 'woods',
 16115: 'spiders',
 2345: 'hanging',
 2289: 'woody',
 52008: 'trawling',
 52009: "hold's",
 11307: 'comically',
 40830: 'localized',
 30568: 'disobeying',
 52010: "'royale",
 40831: "harpo's",
 52011: 'canet',
 19313: 'aileen',
 52012: 'acurately',
 52013: "diplomat's",
 25242: 'rickman',
 6746: 'arranged',
 52014: 'rumbustious',
 52015: 'familiarness',
 52016: "spider'",
 68804: 'hahahah',
 52017: "wood'",
 40833: 'transvestism',
 34702: "hangin'",
 2338: 'bringing',
 40834: 'seamier',
 34703: 'wooded',
 52018: 'bravora',
 16817: 'grueling',
 1636: 'wooden',
 16818: 'wednesday',
 52019: "'prix",
 34704: 'altagracia',
 52020: 'circuitry',
 11585: 'crotch',
 57766: 'busybody',
 52021: "tart'n'tangy",
 14129: 'burgade',
 52023: 'thrace',
 11038: "tom's",
 52025: 'snuggles',
 29114: 'francesco',
 52027: 'complainers',
 52125: 'templarios',
 40835: '272',
 52028: '273',
 52130: 'zaniacs',

### Step 3 — Load the pre-trained model

**What this cell does:**
- `load_model('simple_rnn_imdb.h5')` loads the Simple RNN (trained in `simplernn.ipynb`) together with all its learned weights.
- `model.summary()` prints the architecture to confirm it loaded correctly: Embedding → SimpleRNN → Dense, with 1,313,025 parameters.

**Why it is needed:** this lets us make predictions without re-training. The file `simple_rnn_imdb.h5` must be in the same folder as this notebook.

In [13]:
# Load the pre-trained model with ReLU activation
model = load_model('simple_rnn_imdb.h5')
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

### Step 4 — Inspect the model weights

**What this cell does:** `model.get_weights()` returns a list of NumPy arrays containing every learned weight and bias in the model:
- the embedding table `(10000, 128)`,
- the RNN's input weights, recurrent weights and bias,
- the Dense layer's weights and bias.

**Why it is needed:** it is purely for exploration — to prove that the model has real, trained values. It is not needed for making predictions.

In [14]:
model.get_weights()

[array([[ 0.69172275, -0.6648135 , -0.58848745, ..., -1.0594733 ,
         -1.0943912 , -0.10325934],
        [-0.01867036,  0.01437076, -0.0924966 , ..., -0.0453549 ,
         -0.07134015, -0.00914531],
        [-0.01161603,  0.03310842,  0.06894779, ...,  0.10382418,
         -0.13730131, -0.07176261],
        ...,
        [-0.01507182,  0.02467231, -0.09256832, ...,  0.02101103,
          0.0458507 ,  0.02083372],
        [ 0.01255102, -0.06806055,  0.02516166, ...,  0.03114645,
         -0.04924865, -0.00139675],
        [-0.14248213,  0.10996647, -0.05392867, ..., -0.08769394,
          0.11824714,  0.1233121 ]], dtype=float32),
 array([[-0.01913778,  0.02008036,  0.03849878, ..., -0.00526446,
          0.06914003, -0.19755566],
        [ 0.0852626 ,  0.07021465, -0.10612659, ...,  0.074723  ,
          0.14149873,  0.17267908],
        [ 0.17537591, -0.09544316,  0.00228378, ..., -0.11390357,
          0.18378766, -0.05784669],
        ...,
        [ 0.09404386,  0.08795235,  0.0

### Step 5 — Helper functions: decode and preprocess

**What this cell does:** defines two reusable functions.

**`decode_review(encoded_review)`** — converts a list of integer IDs back to readable text (subtracting 3 because IDs 0–2 are reserved for padding / start / unknown tokens; unknown IDs become `?`).

**`preprocess_text(text)`** — converts a raw user sentence into exactly the format the model expects:
1. `text.lower().split()` — lower-case and split into words.
2. `word_index.get(word, 2) + 3` — look up each word's ID; words not in the dictionary get `2` (unknown). The `+ 3` shifts every ID by 3 to match the offset Keras applied to the training data.
3. `pad_sequences([...], maxlen=500)` — pads/truncates to 500 IDs, the same length used during training.

The result has shape `(1, 500)`: one review, 500 positions.

### Step 5a — Helper functions: decode and preprocess

**The code**

```python
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review
```

**Purpose:** the model only understands integers, but a user types plain text. These two functions convert in both directions:

```
Text  --preprocess_text-->  integers (1, 500)  --> model
Integers  --decode_review-->  readable text
```

---

## Function 1 — `decode_review(encoded_review)`  (integers -> text)

```python
' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])
```

| Part | Meaning |
|---|---|
| `encoded_review` | A list of integer word IDs. |
| `for i in encoded_review` | Loop over each ID. |
| `i - 3` | Undo the offset of 3 that Keras added (IDs 0, 1, 2 are reserved for padding, start and unknown). |
| `reverse_word_index.get(i - 3, '?')` | Look up the word for that ID; if there is none (special tokens), use `'?'`. |
| `' '.join(...)` | Join the words into one sentence separated by spaces. |

- Example: `[1, 14, 22]` -> `'? this film'`.
- It is used only to **read** encoded reviews (for understanding/debugging). It is not called during the prediction below.

---

## Function 2 — `preprocess_text(text)`  (text -> model input)

**Line 1 — `words = text.lower().split()`**
- `lower()` converts to lower case, because the dictionary contains only lower-case words (`"Movie"` -> `"movie"`).
- `split()` cuts the text at spaces into a list of words.
- `"This movie was great"` -> `['this', 'movie', 'was', 'great']`.

**Line 2 — `encoded_review = [word_index.get(word, 2) + 3 for word in words]`**
- `word_index.get(word, 2)` looks up the word's ID in the dictionary used when the model was trained. If the word is not found, it returns `2`.
- `+ 3` shifts every ID by 3 so it matches how Keras encoded the training data (the same offset removed by `i - 3` in `decode_review`).
- Example: `this` -> `11 + 3 = 14`, `film` -> `19 + 3 = 22`.

**Line 3 — `padded_review = sequence.pad_sequences([encoded_review], maxlen=500)`**
- The model was trained on reviews of exactly 500 IDs, so the input must be 500 long.
- Short reviews are padded with `0`s at the **start**; longer ones are cut (the beginning is removed).
- Note the **double brackets** `[encoded_review]`: `pad_sequences` expects a list of sequences, so we pass a list with one review. The result has shape **`(1, 500)`**: a batch of 1 review with 500 IDs, exactly what `model.predict` needs.

**Line 4 — `return padded_review`** hands the array back to the caller.

**Worked example**

```
text           "This movie was great"
words          ['this', 'movie', 'was', 'great']
IDs + 3        [14, 20, 16, 87]      (example values)
padded (1,500) [[0, 0, ..., 0, 14, 20, 16, 87]]    <- 496 zeros, then 4 IDs
```

---

## Limitations to be aware of

- **Punctuation is not removed.** `split()` only splits on spaces, so `"fantastic!"` is looked up with the `!` and is not found, so it is treated as unknown. Removing punctuation first (e.g. with a regex) would improve accuracy.
- **Unknown words get ID 5, not 2.** Missing words use the default `2`, but then `+ 3` is added, so they become `5`, which is actually the ID of a real word (`and`). The training data used `2` for unknown words, so `2` would be the exact match. A more precise version is `word_index[word] + 3 if word in word_index else 2`.
- **Rare words may fall outside the vocabulary.** The model only has 10,000 embedding rows (IDs 0 to 9999). A word whose dictionary index is above about 9,996 gets an ID of 10,000 or more, which does not exist in the table. In training such words were replaced by `2`, so it is safest to map them to `2` too.
- These are simple, tutorial-level preprocessing choices; the model still works well on ordinary reviews.

In [15]:
"""# Step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review"""

"# Step 2: Helper Functions\n# Function to decode reviews\ndef decode_review(encoded_review):\n    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])\n\n# Function to preprocess user input\ndef preprocess_text(text):\n    words = text.lower().split()\n    encoded_review = [word_index.get(word, 2) + 3 for word in words]\n    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)\n    return padded_review"

### Step 5b — Improved preprocessing with a regular expression (regex)

**Why this step:** the original `preprocess_text` only does `text.lower().split()`, which has three weaknesses:
1. **Punctuation sticks to words:** `"fantastic!"` and `"great,"` are not found in the dictionary, so they are treated as unknown.
2. **HTML tags:** IMDB reviews often contain `<br />`, which `split()` would turn into meaningless tokens.
3. **Unknown words get the wrong ID:** `word_index.get(word, 2) + 3` turns an unknown word into `5` (the real word `and`) instead of `2` (the unknown token), and rare words can end up outside the 10,000-word vocabulary.

**What the new code does**

| Piece | Explanation |
|---|---|
| `import re` | Python's regular-expression module. |
| `text.lower()` | Lower-case, because the dictionary is lower-case. |
| `re.sub(r'<[^>]+>', ' ', text)` | Replaces any HTML tag such as `<br />` with a space. `<` starts the tag, `[^>]+` means "one or more characters that are not `>`", `>` ends it. |
| `re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text)` | Extracts the words themselves and ignores all punctuation. |

**Reading the word pattern `[a-z0-9]+(?:'[a-z]+)?`**
- `[a-z0-9]+` -> one or more letters or digits (a word such as `fantastic`).
- `(?:'[a-z]+)?` -> optionally followed by an apostrophe and more letters, so contractions like `don't` and `it's` stay in one piece (they exist in the IMDB dictionary).
- Everything else (`! , . ? ; : - ( )`) is skipped, so `"fantastic!"` becomes `fantastic`.

**Encoding rule (matches how the training data was built)**
- Word found in the dictionary and `index + 3` is below 10,000 -> use `index + 3`.
- Word not found, or outside the 10,000-word vocabulary -> use `2` (the unknown token).

Then the review is padded to 500 as before, giving shape `(1, 500)`.

The code cell below defines `clean_text`, `preprocess_text_regex` and `predict_sentiment_regex`, and compares the old and new preprocessing on a review with punctuation. The original functions are left untouched.

In [16]:
import re

max_features = 10000   # vocabulary size used when the model was trained
max_len = 500          # review length used when the model was trained

def clean_text(text):
    #Lower-case, remove HTML tags and punctuation, return a list of words.
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)               # remove tags such as <br />
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text)  # keep words (and contractions) only

def preprocess_text_regex(text):
    words = clean_text(text)
    encoded_review = []
    for word in words:
        idx = word_index.get(word)
        if idx is None or idx + 3 >= max_features:
            encoded_review.append(2)          # unknown / out-of-vocabulary token
        else:
            encoded_review.append(idx + 3)    # same +3 offset used in training
    return sequence.pad_sequences([encoded_review], maxlen=max_len)


In [20]:
def predict_sentiment_regex(review):
    preprocessed_input = preprocess_text_regex(review)
    prediction = model.predict(preprocessed_input)
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    return sentiment, prediction[0][0]

# Compare the old and the new preprocessing on a review that has punctuation and HTML
test_review = "This movie was fantastic! The acting was great, and the plot was thrilling.Don't miss it."

print("Old split() words :", test_review.lower().split())
print("New regex words   :", clean_text(test_review))

sentiment, score = predict_sentiment_regex(test_review)
print(f"Sentiment: {sentiment}")
print(f"Prediction Score: {score}")

Old split() words : ['this', 'movie', 'was', 'fantastic!', 'the', 'acting', 'was', 'great,', 'and', 'the', 'plot', 'was', "thrilling.don't", 'miss', 'it.']
New regex words   : ['this', 'movie', 'was', 'fantastic', 'the', 'acting', 'was', 'great', 'and', 'the', 'plot', 'was', 'thrilling', "don't", 'miss', 'it']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
Sentiment: Positive
Prediction Score: 0.8675194382667542


in case of using 5a hence commented as we use regex one

### Step 6 — Prediction function

**What this cell does:** defines `predict_sentiment(review)` which does the whole job in one call:
1. Preprocesses the review text with `preprocess_text`.
2. Runs `model.predict(...)`, which returns a probability between 0 and 1 (shape `(1, 1)`).
3. Applies a **0.5 threshold**: greater than 0.5 → `'Positive'`, otherwise `'Negative'`.
4. Returns both the sentiment label and the raw score.

**Why it is needed:** wrapping the steps in a function makes it easy to classify any number of reviews, and the same logic is reused in the Streamlit app (`main.py`).

In [18]:
### Prediction  function

"""def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'

    return sentiment, prediction[0][0]"""



"def predict_sentiment(review):\n    preprocessed_input=preprocess_text(review)\n\n    prediction=model.predict(preprocessed_input)\n\n    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'\n    \n    return sentiment, prediction[0][0]"

### Step 7 — Try it on an example review

**What this cell does:**
- Defines an example review: *"This movie was fantastic! The acting was great and the plot was thrilling."*
- Calls `predict_sentiment` and prints the review, the sentiment and the score.

**Output:** `Sentiment: Positive`, `Prediction Score: 0.81` — the model is about 81% confident the review is positive. The closer the score is to 1 the more positive it is; closer to 0 means more negative.

**Tip:** change `example_review` to your own text to test the model. Note that punctuation attached to words (e.g. `fantastic!`) is not stripped by this simple preprocessing, so such words may map to the unknown token.

In [19]:
# Step 4: User Input and Prediction
# Example review for prediction
"""example_review = "This movie was fantastic.The acting was great and the plot was thrilling."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')"""

'example_review = "This movie was fantastic.The acting was great and the plot was thrilling."\n\nsentiment,score=predict_sentiment(example_review)\n\nprint(f\'Review: {example_review}\')\nprint(f\'Sentiment: {sentiment}\')\nprint(f\'Prediction Score: {score}\')'